Goal: Predict the probability by race of getting hired in a company

Evaluation criteria: Prioritise accuracy to determine the probability of getting hired in a company by race

Model description: We train a logistic regression model using only the race feature encoded via one-hot encoding to directly estimate the probability of being “hired” (proxy via income >50K). We convert income to a binary target, then evaluate the model using 5-fold stratified cross‐validation and log‐loss to prioritize probabilistic accuracy. After reporting the mean log‐loss, we fit the model on the full dataset to compute the estimated hire probability for each race category.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Load data
df = pd.read_csv("../input/adult_reconstruction.csv")

# Prepare features and target
X = pd.get_dummies(df[["race"]], drop_first=False)
y = (df["income"] > 50000).astype(int)

# 5-fold stratified cross-validation with log-loss
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(max_iter=1000)
scores = cross_val_score(model, X, y, cv=cv, scoring="neg_log_loss")
mean_log_loss = -scores.mean()
print(f"Mean Log-loss: {mean_log_loss:.5f}")

# Fit on full data and compute hire probability by race
model.fit(X, y)
races = sorted(df["race"].unique())
race_df = pd.DataFrame({"race": races})
X_race = pd.get_dummies(race_df["race"], drop_first=False).reindex(
    columns=X.columns, fill_value=0
)
probs = model.predict_proba(X_race)[:, 1]
result = pd.DataFrame({"race": races, "hire_probability": probs})
print("\nEstimated hire probability by race:")
print(result)

Mean Log-loss: 0.54329

Estimated hire probability by race:
                 race  hire_probability
0  Amer-Indian-Eskimo          0.166808
1  Asian-Pac-Islander          0.166808
2               Black          0.166808
3               Other          0.166808
4               White          0.166808
